# EmotWen — Step 3: Evaluation

Evaluates the SFT-trained model on the held-out `eval_200` set.

**Metrics computed:**
- Sentence count distribution (2–5 target)
- Advice rate (regex blocklist)
- Emotion alignment (j-hartmann classifier)
- Empathy score (LLM-as-judge, optional — requires OpenAI or Anthropic API key)
- Perplexity proxy (mean NLL/token on val set)
- `'Let me explain:'` exception accuracy

**Decision gate:** if >15% of responses exceed 5 sentences → run `04_grpo.ipynb`

**Prerequisites:** Run `02_sft_train.ipynb` first.

In [ ]:
%%capture
import os, importlib.util, sys
!pip install --upgrade -qqq uv
if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = 'numpy'; _pil = 'pillow'
    !uv pip install -qqq \
        'torch==2.8.0' 'triton>=3.3.0' {_numpy} {_pil} bitsandbytes 'xformers==0.0.32.post2' \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth'
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers 'trl==0.22.2' unsloth unsloth_zoo
!uv pip install 'transformers==5.2.0'
!uv pip install --no-build-isolation flash-linear-attention 'causal_conv1d==1.6.0'
!uv pip install wandb datasets nltk

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/emotwen-3.5-finetune'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import nltk
nltk.download('punkt_tab', quiet=True)

In [ ]:
import wandb
wandb.login()

In [ ]:
# ── Optional: set API keys for LLM-as-judge ──────────────────────────────────
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

In [ ]:
# ── Config overrides ──────────────────────────────────────────────────────────
CONFIG = {
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'eval_post_sft_v1',

    # Path to SFT adapter (stage 2)
    'adapter_path': os.path.join(REPO_DIR, 'outputs', 'sft_stage2'),

    # Eval samples (up to 200)
    'n_samples': 200,

    # LLM judge: set to None to skip, or 'gpt-4o-mini' / 'claude-haiku-4-5-20251001'
    'judge_model': None,

    # GRPO trigger threshold
    'grpo_trigger_pct': 0.15,
}

In [ ]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
from src.evaluate import run

results = run(CONFIG)

print('\n── Summary ──────────────────────────────────────────────')
print(f'  Responses in range (2–5 sentences): {results["pct_in_range"]:.1%}')
print(f'  Responses >5 sentences:             {results["pct_over_5"]:.1%}')
print(f'  Advice rate:                        {results["advice_rate"]:.1%}')
print(f'  Emotion alignment:                  {results["emotion_alignment"]:.1%}')
print(f'  Perplexity (mean NLL/token):        {results["perplexity"]:.4f}')
print(f'  LLM empathy score (reflection):     {results["llm_reflection_avg"]}')
print(f'  LLM no-advice score:                {results["llm_no_advice_avg"]}')
print(f'  "Let me explain:" accuracy:         {results["let_me_explain_ok_rate"]}')
print()
print(f'  ➜ Run GRPO: {"YES ⚠" if results["grpo_needed"] else "NO ✓"}')